# 4. Honest confidence, and the ternary question

**Calibration:** when a model says "90% sure", is it right about 90% of the time? Answers are sorted into 15
bands by confidence; the gaps between average confidence and accuracy are combined into one number, the
**expected calibration error (ECE)**. Lower is better.

**Temperature scaling** fixes a uniform over- or under-confidence with a single number T. To avoid tuning on the
reported photos, T is fitted on the even-numbered photos and ECE is reported on the odd-numbered ones.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # run from the repository root
sys.path.insert(0, "src")
import numpy as np
from IPython.display import Image, display

In [2]:
T = json.load(open("evidence/rescore/rescore.json"))["temperature"]
print(f"{'model':<14}{'ECE before':>11}{'ECE after':>11}{'T':>8}")
for k in ["deit_s_fp32"] + [f"{r}-s{s}" for r in "ABCD" for s in (0, 1, 2)]:
    t = T[k]; print(f"{k:<14}{t['ece_odd_before']:>11.2f}{t['ece_odd_after']:>11.2f}{t['T']:>8.3f}")

model          ECE before  ECE after       T
deit_s_fp32          8.56       3.40   0.833
A-s0                 4.43       4.08   0.874
A-s1                 4.39       4.08   0.873
A-s2                 4.34       4.11   0.873
B-s0                 5.31       4.03   0.859
B-s1                 5.35       4.00   0.860
B-s2                 5.65       3.74   0.860
C-s0                 4.56       4.00   0.874
C-s1                 4.61       4.32   0.870
C-s2                 4.60       4.22   0.871
D-s0                11.84       3.14   0.802
D-s1                12.00       2.97   0.801
D-s2                12.05       2.92   0.801


**Reading it:** the distilled model (D) is the worst calibrated before correction (about 12) and the best after
(about 3), so its problem is a simple uniform offset that one number removes.

## 4.2 Is ternary specially weak on noise, or just a weaker model?

A model that is better on clean photos is usually better on damaged ones by a predictable amount
(Miller et al., 2021). Seven DeiT-Small models of slightly different quality (route A at five group sizes, plus
routes A and C) draw that trend line. If ternary falls below the line on noise, it is specially weak on noise.

The rule was fixed before looking: "below the line" only if the 95% range excludes zero.

In [3]:
o6 = json.load(open("evidence/rescore/o6_line.json"))
for s in (0, 1, 2):
    r = o6[f"noise_s{s}"]
    print(f"seed {s}: ternary sits {r['residual']:+.2f} pts from the line on noise, 95% range [{r['ci'][0]:+.2f}, {r['ci'][1]:+.2f}]")

seed 0: ternary sits -5.03 pts from the line on noise, 95% range [-8.82, -1.81]
seed 1: ternary sits -4.56 pts from the line on noise, 95% range [-9.02, -0.61]
seed 2: ternary sits -4.40 pts from the line on noise, 95% range [-8.58, -1.14]


**Reading it:** ternary sits 4–5 points below the line in every seed, and each 95% range excludes zero. But
with a stricter range for testing three seeds (Bonferroni, 98.3%) two of three seeds include zero, and noise is
a damage type where the trend often breaks down (Miller et al.). So this is reported as consistent, not proven.
The full check is in `results.md`.